**Author:** Amanda Baright

**Date:** 04.21.2026

**Purpose:** ST 554 Homework 10

You are tasked with using spark structured streaming to deal with streaming data!

# Part 1: Creating Streaming Data Using `rate`

We first want to setup a data stream using the "rate" formula from class.

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
rateDF = spark.readStream.format("rate").load()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/21 19:58:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/21 19:58:43 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
rateDF # look at the rateDF object

DataFrame[timestamp: timestamp, value: bigint]

We then want to set up a sequence of actions using appropriate functions from `pyspark.sql.functions` that uses the `rate` data to find the square root of the rate 'value' and find mod 4 of the rate 'value'. Here we will use the `pyspark.sql.functions` of `col`, `sqrt`, and `expr`.

In [4]:
from pyspark.sql.functions import col, sqrt, expr

transformedDF = rateDF.select(
    col("timestamp"), # grab the 'timestamp' column
    col("value"), # grab the 'value' column
    # Find the square root of the rate 'value' 
    sqrt(col("value")).alias("sqrt_value"),
    # Find mod 4 of the rate 'value' 
    # We can use expr() for SQL-like syntax for the modulo operator
    expr("value % 4").alias("mod_4_value")
)

We now want to output this by creating a `writeStream` that writes to 'memory' (`format("memory")`). We will give the query a name (`queryName("...")`) and start it. We will let it run for about 30 seconds and then we will stop the query.

In [6]:
query = transformedDF.writeStream \
    .outputMode("append") \
    .format("memory") \
    .queryName("homework_table") \
    .start()

26/04/21 20:10:04 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-e45273af-487f-43d5-a079-015bc4435527. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/21 20:10:04 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Here we will use the `time` function to `sleep` for 30 seconds and then run `query.stop()` to stope the query.

In [7]:
import time
time.sleep(30)
query.stop()

26/04/21 20:10:55 WARN DAGScheduler: Failed to cancel job group b1c4f319-bccb-4851-ad99-bb13d8a9a8d7. Cannot find active jobs for it.
26/04/21 20:10:55 WARN DAGScheduler: Failed to cancel job group b1c4f319-bccb-4851-ad99-bb13d8a9a8d7. Cannot find active jobs for it.


We then want to output the entire table stored int he query name using `spark.sql("select * from homework_table").show()`.

In [8]:
spark.sql("SELECT * FROM homework_table").show()

+--------------------+-----+------------------+-----------+
|           timestamp|value|        sqrt_value|mod_4_value|
+--------------------+-----+------------------+-----------+
|2026-04-21 20:10:...|    0|               0.0|          0|
|2026-04-21 20:10:...|    1|               1.0|          1|
|2026-04-21 20:10:...|    2|1.4142135623730951|          2|
|2026-04-21 20:10:...|    3|1.7320508075688772|          3|
|2026-04-21 20:10:...|    4|               2.0|          0|
|2026-04-21 20:10:...|    5|  2.23606797749979|          1|
|2026-04-21 20:10:...|    6| 2.449489742783178|          2|
|2026-04-21 20:10:...|    7|2.6457513110645907|          3|
|2026-04-21 20:10:...|    8|2.8284271247461903|          0|
|2026-04-21 20:10:...|    9|               3.0|          1|
|2026-04-21 20:10:...|   10|3.1622776601683795|          2|
|2026-04-21 20:10:...|   11|   3.3166247903554|          3|
|2026-04-21 20:10:...|   12|3.4641016151377544|          0|
|2026-04-21 20:10:...|   13| 3.605551275

# Part 2 - Using data from a CSV with a Pipeline

This next part will use the six `bikeDetails` sub datasets that were uploaded into the JupyterLab environment. The one named `bikeDetails_for_fit.csv` will be read in as a spark SQL data frame. With this spark SQL data frame we will first use a `SQLTransformer` with the following statment to do some log transforms, rename a variable, and create a dummy variable from a categorical variable.

In [9]:
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import SQLTransformer, VectorAssembler

# Initialize Spark Session
spark = SparkSession.builder.appName("HW10_Part2").getOrCreate()

static_df = spark.read.csv("bikeDetails_for_fit.csv", header=True, inferSchema=True)

26/04/21 20:21:13 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [11]:
static_df.head(5) # Look at first five rows

[Row(name='Royal Enfield Classic 350', selling_price=175000, year=2019, seller_type='Individual', owner='1st owner', km_driven=350, ex_showroom_price=None),
 Row(name='Honda Dio', selling_price=45000, year=2017, seller_type='Individual', owner='1st owner', km_driven=5650, ex_showroom_price=None),
 Row(name='Royal Enfield Classic Gunmetal Grey', selling_price=150000, year=2018, seller_type='Individual', owner='1st owner', km_driven=12000, ex_showroom_price=148114),
 Row(name='Yamaha Fazer FI V 2.0 [2016-2018]', selling_price=65000, year=2015, seller_type='Individual', owner='1st owner', km_driven=23000, ex_showroom_price=89643),
 Row(name='Yamaha SZ [2013-2014]', selling_price=20000, year=2011, seller_type='Individual', owner='2nd owner', km_driven=21000, ex_showroom_price=None)]

In [12]:
# Now run the SQLTransformer code
sql_statement = """
SELECT log(selling_price) as label, year, log(km_driven) as log_km_driven,
CASE WHEN Owner = '1st owner' THEN 1 ELSE 0 END AS one_owner
FROM __THIS__
"""
sql_trans = SQLTransformer(statement=sql_statement)

We then want to use a `VectorAssembler` to create a `features` column. The `features` column will include the `year`, `log_km_driven`, and `one_owner` variable.

In [13]:
assembler = VectorAssembler(
    inputCols=["year", "log_km_driven", "one_owner"], 
    outputCol="features"
)

We then want to create a `Pipeline` with the two steps above, meaning using the `SQLTransformer` and then the `VectorAssembler`.

In [14]:
pipeline = Pipeline(stages=[sql_trans, assembler])

We then want to fit this pipeline to the SQL data frame and save this as an object.

In [15]:
fitted_pipeline = pipeline.fit(static_df)

We now want to set up a read stream to look for csv files placed into a folder (the five `bikeDetails_add*.csv` files). We will create a folder called `HW10_streaming_folder` that will be used later on. When a csv comes in, we will want to transform it using the pipeline's `.transfrom()` method. For this to work, we will need a schema to set up the `readStream`. Here we can use the SQL data frame's schema from above using the `.schema` attribute. We also need to note that each csv file contains a header, which we will take into account with the `.option("header","true")` statement.

In [16]:
streaming_df = spark.readStream \
    .schema(static_df.schema) \
    .option("header", "true") \
    .csv("HW10_streaming_folder/")

Now we can use the `.transform()` method to apply the logic we used earlier on the incoming data.

In [17]:
transformed_stream = fitted_pipeline.transform(streaming_df)

We will write the output to the 'console' using the 'append' mode.

We first will want to make sure that `HW10_streaming_folder` is empty. Then we will start the query. We will place the files into the folder one at a time and should see the output appear below the query. Once all five files are done, we will stop the query.

In [18]:
query = transformed_stream.writeStream \
    .outputMode("append") \
    .format("console") \
    .start()

26/04/21 20:39:10 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-9e07b6b7-5754-4aee-b500-5ecb46959b64. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/21 20:39:10 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+------------------+----+------------------+---------+--------------------+
|             label|year|     log_km_driven|one_owner|            features|
+------------------+----+------------------+---------+--------------------+
| 8.987196820661973|2003|10.887436932884098|        1|[2003.0,10.887436...|
|11.156250521031495|2018| 9.615805480084347|        1|[2018.0,9.6158054...|
|10.819778284410283|2016| 8.987196820661973|        1|[2016.0,8.9871968...|
| 10.46310334047155|2015|10.582738627903963|        1|[2015.0,10.582738...|
| 9.903487552536127|2006|11.225243392518447|        1|[2006.0,11.225243...|
|10.819778284410283|2012|10.239959789157341|        1|[2012.0,10.239959...|
| 10.51867319162636|2008| 11.03488966402723|        1|[2008.0,11.034889...|
|11.141861783579396|2018| 9.392661928770137|        1|[2018.0,9.3926619...|
|10.239959789157341|2012| 10.81975828421028|        1|[2012.0,10.81

In [19]:
query.stop()

26/04/21 20:39:42 WARN DAGScheduler: Failed to cancel job group 5646fd99-4b68-4f87-9d5a-0f55dfe58f59. Cannot find active jobs for it.
26/04/21 20:39:42 WARN DAGScheduler: Failed to cancel job group 5646fd99-4b68-4f87-9d5a-0f55dfe58f59. Cannot find active jobs for it.


This process will set us up for using a fitted model to obtain predictions on streaming data.